# Notebook 2: AC2001 vs TWAP vs VWAP — Monte Carlo Comparison

Runs a paired Monte Carlo simulation comparing implementation shortfall (IS) for three strategies:

- **AC2001**: optimal hyperbolic trajectory
- **TWAP**: uniform liquidation (risk-neutral optimal)
- **VWAP**: volume-weighted, follows a U-shaped intraday profile

All three strategies execute the same position over the same horizon and face the same random price shocks in each simulation (paired comparison).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from src.execution_sim import compare_strategies, slippage_summary, intraday_u_shape
from src.almgren_chriss import optimal_trajectory, twap_trajectory
from src.execution_sim import vwap_holdings

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Model parameters
X     = 1_000_000   # shares
T     = 1.0         # 1 day
N     = 390         # 1-minute intervals
sigma = 0.02
gamma = 2.5e-7
eta   = 2.5e-6
lam   = 1e-6        # risk aversion
S0    = 100.0
N_SIMS = 2000
SEED   = 42

## 1. Trade Schedules

In [ ]:
t, h_ac   = optimal_trajectory(X, T, N, sigma, gamma, eta, lam)
h_twap    = twap_trajectory(X, N, T)
h_vwap    = vwap_holdings(X, intraday_u_shape(N))

# Compute trade sizes
n_ac   = -np.diff(h_ac)
n_twap = -np.diff(h_twap)
n_vwap = -np.diff(h_vwap)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t, h_ac   / 1e6, label='AC2001',     lw=2)
axes[0].plot(t, h_twap / 1e6, label='TWAP',       lw=2, ls='--')
axes[0].plot(t, h_vwap / 1e6, label='VWAP (U)',   lw=2, ls=':')
axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('Remaining position (M shares)')
axes[0].set_title('Holdings Trajectories')
axes[0].legend()

t_mid = t[:-1] + (T/N)/2
axes[1].plot(t_mid * 390, n_ac   / 1e3, label='AC2001', lw=1.5)
axes[1].plot(t_mid * 390, n_twap / 1e3, label='TWAP',   lw=1.5, ls='--')
axes[1].plot(t_mid * 390, n_vwap / 1e3, label='VWAP',   lw=1.5, ls=':')
axes[1].set_xlabel('Interval (minutes from open)')
axes[1].set_ylabel('Trade size (k shares)')
axes[1].set_title('Trade Schedules')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/fig_schedules.png', dpi=150)
plt.show()

## 2. Monte Carlo Implementation Shortfall

In [ ]:
print(f"Running {N_SIMS:,} simulations...")
stats = compare_strategies(
    X=X, T=T, N=N, sigma=sigma, gamma=gamma, eta=eta, lam=lam,
    S0=S0, n_sims=N_SIMS, seed=SEED,
)
print(slippage_summary(stats))

In [ ]:
# Build a tidy DataFrame for seaborn
rows = []
for name, s in stats.items():
    for val in s.is_bps_all:
        rows.append({'Strategy': name, 'IS (bps)': val})
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Box plot
sns.boxplot(data=df, x='Strategy', y='IS (bps)', ax=axes[0],
            order=['AC2001', 'TWAP', 'VWAP'], width=0.5)
axes[0].set_title(f'IS Distribution ({N_SIMS:,} simulations)')
axes[0].set_ylabel('Implementation Shortfall (bps)')

# KDE overlay
for name, color in zip(['AC2001', 'TWAP', 'VWAP'], ['steelblue', 'tomato', 'seagreen']):
    sns.kdeplot(stats[name].is_bps_all, ax=axes[1], label=name, color=color, fill=True, alpha=0.25)
axes[1].set_xlabel('Implementation Shortfall (bps)')
axes[1].set_ylabel('Density')
axes[1].set_title('IS Density by Strategy')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/fig_is_comparison.png', dpi=150)
plt.show()

## 3. Summary Statistics

In [ ]:
summary = []
for name, s in stats.items():
    arr = s.is_bps_all
    summary.append({
        'Strategy':        name,
        'Mean IS (bps)':   round(s.mean_is_bps, 2),
        'Std IS (bps)':    round(s.std_is_bps, 2),
        'P5 (bps)':        round(np.percentile(arr, 5), 2),
        'P95 (bps)':       round(np.percentile(arr, 95), 2),
        'Sharpe IS':       round(s.sharpe_is, 3),
    })

pd.DataFrame(summary).set_index('Strategy')

## 4. Interpretation

- **TWAP** minimises E[C] but carries timing risk: a large variance of IS.
- **AC2001** with $\lambda > 0$ accepts slightly higher E[C] in exchange for lower Var[C], improving the *risk-adjusted* IS.
- **VWAP** front-loads at open and close (U-shape profile), which may align with liquidity but is not necessarily optimal for a given $\lambda$.

The Sharpe IS ratio measures how consistently AC reduces slippage relative to TWAP.  A positive value indicates it reliably saves bps on a per-simulation basis.